In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torchvision.transforms as T
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, roc_auc_score, roc_curve, auc
from sklearn.preprocessing import label_binarize
import medmnist
from medmnist import INFO
from tqdm import tqdm
from cnn import CNN
from resnet import ResNet18
from vit import VisionTransformer
import torchvision.models as models
from training_evalutation_utils import * 
from plotting import * 

/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [ ]:
# Dataset configuration
data_flag = 'tissuemnist'
info = INFO[data_flag]
task = info['task']
n_channels = info['n_channels']
n_classes = len(info['label'])
class_names = list(info['label'].values())

print(f"\nDataset: {data_flag}")
print(f"Task: {task}")
print(f"Input channels: {n_channels}")
print(f"Number of classes: {n_classes}")
print(f"Classes: {info['label']}")

DataClass = getattr(medmnist, info['python_class'])

# Data transformations for models trained from scratch
transform_scratch = T.Compose([
    T.ToTensor(),
    T.Normalize(mean=[0.5], std=[0.5])
])

# Data transformations for transfer learning (ViT expects 3 channels and specific normalization)
transform_transfer = T.Compose([
    T.Grayscale(num_output_channels=3),
    T.Resize(224),  # ViT expects 224x224 images
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # ImageNet stats
])

# Load datasets for scratch models
train_dataset_scratch = DataClass(split='train', transform=transform_scratch, download=True, size=64)
val_dataset_scratch = DataClass(split='val', transform=transform_scratch, download=True, size=64)
test_dataset_scratch = DataClass(split='test', transform=transform_scratch, download=True, size=64)

# Load datasets for transfer learning
train_dataset_transfer = DataClass(split='train', transform=transform_transfer, download=True, size=64)
val_dataset_transfer = DataClass(split='val', transform=transform_transfer, download=True, size=64)
test_dataset_transfer = DataClass(split='test', transform=transform_transfer, download=True, size=64)


Dataset: tissuemnist
Task: multi-class
Input channels: 1
Number of classes: 8
Classes: {'0': 'Collecting Duct, Connecting Tubule', '1': 'Distal Convoluted Tubule', '2': 'Glomerular endothelial cells', '3': 'Interstitial endothelial cells', '4': 'Leukocytes', '5': 'Podocytes', '6': 'Proximal Tubule Segments', '7': 'Thick Ascending Limb'}


100%|██████████| 555M/555M [02:13<00:00, 4.17MB/s] 


In [6]:
# Create data loaders
batch_size = 32

train_loader_scratch = DataLoader(train_dataset_scratch, batch_size=batch_size, shuffle=True)
val_loader_scratch = DataLoader(val_dataset_scratch, batch_size=batch_size, shuffle=False)
test_loader_scratch = DataLoader(test_dataset_scratch, batch_size=batch_size, shuffle=False)

train_loader_transfer = DataLoader(train_dataset_transfer, batch_size=batch_size, shuffle=True)
val_loader_transfer = DataLoader(val_dataset_transfer, batch_size=batch_size, shuffle=False)
test_loader_transfer = DataLoader(test_dataset_transfer, batch_size=batch_size, shuffle=False)

print(f"\nTraining samples: {len(train_dataset_scratch)}")
print(f"Validation samples: {len(val_dataset_scratch)}")
print(f"Test samples: {len(test_dataset_scratch)}")


Training samples: 165466
Validation samples: 23640
Test samples: 47280


In [1]:
print("\n" + "=" * 70)
print("MODEL 1: ResNet50 (Transfer Learning)")
print("=" * 70)

resnet50 = models.resnet50(pretrained=True)
# Modify final layer for our number of classes
resnet50.fc = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(resnet50.fc.in_features, 512), 
    nn.ReLU(), 
    nn.Linear(512, 256), 
    nn.ReLU(), 
    nn.Dropout(0.5),
    nn.Linear(256, n_classes)
)
resnet50 = resnet50.to(device)

# Option to freeze backbone (uncomment to only train classifier)
for param in resnet50.parameters():
    param.requires_grad = False
for param in resnet50.fc.parameters():
    param.requires_grad = True

resnet_optimizer = torch.optim.AdamW(resnet50.parameters(), lr=1e-4, weight_decay=1e-2)
criterion = nn.CrossEntropyLoss()

train_losses_resnet, val_losses_resnet, val_accs_resnet = fit(
    resnet50, train_loader, val_loader, criterion, resnet_optimizer,
    epochs=50, device=device, model_name="ResNet50_tl", patience=5
)

# Evaluate ResNet50
resnet_acc, resnet_preds, resnet_labels, resnet_probs = evaluate_model(resnet50, test_loader, device)
print(f"\n✓ ResNet50 Test Accuracy: {resnet_acc:.4f}")

plot_training_history(train_losses_resnet, val_losses_resnet, val_accs_resnet, 
                     "ResNet50", save_path="resnet50_losses.png")
plot_confusion_matrix(resnet_labels, resnet_preds, class_names, 
                     "ResNet50", save_path="resnet50_confusion.png")
resnet_auc = plot_roc_curves(resnet_labels, resnet_probs, n_classes, class_names,
                             "ResNet50", save_path="resnet50_roc.png")


MODEL 1: ResNet50 (Transfer Learning)


NameError: name 'models' is not defined